# EvalRX on a Colab TPU: launch from a prebuilt image

[Open in Colab](https://colab.research.google.com/github/evalvitals/evalrx/blob/ruinan/examples/colab/tpu/launch.ipynb)

This notebook installs nothing. It restores a **prebuilt EvalRX image**, which holds a Python 3.12 interpreter, every locked package (jax, libtpu, gemma, ...), the `agy` agent, the EvalRX source and the benchmark datasets, already installed, and then runs the complete EvalRX workflow on Gemma 4 E2B. The image is made once by [`setup.ipynb`](https://github.com/evalvitals/evalrx/blob/ruinan/examples/colab/tpu/setup.ipynb). Restoring it unpacks one tar to `/content`. It does not use pip, uv, git, GitHub or PyPI, and it does not import or change the Colab kernel's own packages. The same image therefore runs on public Colab, on Google-internal Colab, and on later Colab images.

1. Select a **TPU** runtime.
2. Put the image tar and its `.sha256` file where this runtime can read them: Google Drive, a GCS bucket, an https:// URL, or `/content`. Do the same for the weights tar unless the runtime can read `gs://gemma-data`.
3. Add `GEMINI_API_KEY` to Colab Secrets and allow notebook access, or enter the key when asked. The `agy` agent calls the Gemini API; it is the only network service the run needs.
4. Fill in section 1 and run all cells.

## 1. Image, weights and task

Only this cell needs editing. `DATASET` selects BBH word sorting or CRUXEval output prediction; both use 128 cases, seed 0 and a 64/64 EXPLORE/CONFIRM split.

In [ ]:
import os
# Paths: Drive (/content/drive/MyDrive/...), gs://bucket/..., https://..., or a local file.
# The checksum is read from <path>.sha256 next to each tar, or set it below.
EVALRX_IMAGE = os.environ.get('EVALRX_IMAGE', '')      # evalrx-colab-tpu-image-<commit>.tar (setup.ipynb)
EVALRX_IMAGE_SHA256 = os.environ.get('EVALRX_IMAGE_SHA256', '')
EVALRX_WEIGHTS = os.environ.get('EVALRX_WEIGHTS', '')  # evalrx-colab-weights-tpu-gemma-4-e2b.tar; '' = read gs://gemma-data
EVALRX_WEIGHTS_SHA256 = os.environ.get('EVALRX_WEIGHTS_SHA256', '')
DATASET = os.environ.get('EVALRX_DATASET', 'bbh_word_sorting')  # or 'cruxeval_output'
RUN_TAG = 'launch'
assert EVALRX_IMAGE, 'set EVALRX_IMAGE to the image tar made by setup.ipynb'

## 2. Restore the environment

The tar is checked against its SHA-256 while it is unpacked, so a damaged or partial copy is rejected. Rerunning the cell after a restore takes no time. When `EVALRX_WEIGHTS` is set, the model loads from it, and `gs://gemma-data` is not read.

In [ ]:
import hashlib, json, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

CONTENT = Path('/content')
ENV, REPO = CONTENT / 'evalrx-env', CONTENT / 'evalrx'
DEMO_ROOT = CONTENT / 'evalrx_examples'
# The Colab kernel exports settings for its own Python (PYTHONPATH, UV_*, PIP_*);
# they must not reach the image's Python, which every later cell runs.
for key in [k for k in os.environ if k.startswith(('UV_', 'PIP_')) or k in ('PYTHONPATH', 'PYTHONHOME')]:
    del os.environ[key]
if any(p.startswith('/content/drive/') for p in (EVALRX_IMAGE, EVALRX_WEIGHTS)) and not Path('/content/drive/MyDrive').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')

def open_source(src):
    """A binary stream of a local or Drive path, an https:// URL, or a gs:// object."""
    if src.startswith(('http://', 'https://')):
        return urllib.request.urlopen(src)
    if src.startswith('gs://'):
        tool = shutil.which('gcloud') or shutil.which('gsutil')
        if tool:
            cmd = [tool, 'storage', 'cat', src] if tool.endswith('gcloud') else [tool, 'cat', src]
            return subprocess.Popen(cmd, stdout=subprocess.PIPE).stdout
        # No Cloud SDK (e.g. Colab TPU runtimes): the kernel's GCS client, with your
        # credentials (google.colab.auth.authenticate_user()), else anonymous.
        from google.cloud import storage
        bucket, _, name = src[len('gs://'):].partition('/')
        try:
            blob = storage.Client(project=os.environ.get('GOOGLE_CLOUD_PROJECT', '_')).bucket(bucket).blob(name)
            blob.reload()
        except Exception:
            blob = storage.Client.create_anonymous_client().bucket(bucket).blob(name)
        return blob.open('rb')
    return open(src, 'rb')

def expected_sha256(src, given):
    if given:
        return given.strip().split()[0]
    try:
        with open_source(src + '.sha256') as f:
            return f.read().decode().split()[0]
    except Exception as exc:
        raise RuntimeError(f'no checksum for {src}: put {Path(src).name}.sha256 next to it, '
                           'or set the *_SHA256 variable in section 1') from exc


def restore(src, sha256, staging):
    """Stream src through SHA-256 into `tar -x` in staging; raise if the checksum differs."""
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir(parents=True)
    digest, done, start, step = hashlib.sha256(), 0, time.time(), 1 << 30
    tar = subprocess.Popen(['tar', '-x', '-C', str(staging)], stdin=subprocess.PIPE)
    with open_source(src) as stream:
        while True:
            chunk = stream.read(8 << 20)
            if not chunk:
                break
            digest.update(chunk)
            tar.stdin.write(chunk)
            if (done + len(chunk)) // step > done // step:
                print(f'  {(done + len(chunk)) / 1e9:.0f} GB, {time.time() - start:.0f} s', flush=True)
            done += len(chunk)
    tar.stdin.close()
    if tar.wait():
        raise RuntimeError(f'tar failed on {src}')
    if digest.hexdigest() != sha256:
        shutil.rmtree(staging)
        raise RuntimeError(f'checksum mismatch for {src}: got {digest.hexdigest()}, expected {sha256}. Copy it again.')
    print(f'  verified sha256 {sha256[:16]}..., {done / 1e9:.1f} GB in {time.time() - start:.0f} s')


image_sha = expected_sha256(EVALRX_IMAGE, EVALRX_IMAGE_SHA256)
marker = ENV / '.image-sha256'
if marker.exists() and marker.read_text().strip() == image_sha:
    print('image already restored:', image_sha[:16])
else:
    print('restoring', EVALRX_IMAGE)
    staging = CONTENT / '.evalrx-image-staging'
    restore(EVALRX_IMAGE, image_sha, staging)
    for name in ('evalrx', 'evalrx-env'):  # fixed paths: the environment refers to them
        shutil.rmtree(CONTENT / name, ignore_errors=True)
        (staging / name).rename(CONTENT / name)
    shutil.rmtree(staging)
    marker.write_text(image_sha + '\n')

model_path = ''
if EVALRX_WEIGHTS:
    weights_sha = expected_sha256(EVALRX_WEIGHTS, EVALRX_WEIGHTS_SHA256)
    weights_root = CONTENT / 'evalrx-weights'
    wmarker = weights_root / '.sha256'
    if not (wmarker.exists() and wmarker.read_text().strip() == weights_sha):
        print('restoring', EVALRX_WEIGHTS)
        shutil.rmtree(weights_root, ignore_errors=True)
        restore(EVALRX_WEIGHTS, weights_sha, weights_root)
        wmarker.write_text(weights_sha + '\n')
    (model_dir,) = [p for p in weights_root.iterdir() if p.is_dir()]
    model_path = str(model_dir)

# Datasets frozen into the image; agy reads GEMINI_API_KEY when its provider is "gemini".
shutil.copytree(REPO / '.bundle/data', DEMO_ROOT / 'data', dirs_exist_ok=True)
settings = Path.home() / '.gemini/antigravity-cli/settings.json'
config = json.loads(settings.read_text()) if settings.exists() else {}
settings.parent.mkdir(parents=True, exist_ok=True)
settings.write_text(json.dumps({**config, 'modelProvider': 'gemini'}, indent=2))

os.environ.update(EVALRX_ENV=str(ENV), EVALRX_DEMO_ROOT=str(DEMO_ROOT), EVALRX_DATASET=DATASET,
                  EVALRX_RUN_TAG=RUN_TAG, EVALRX_MODEL_PATH=model_path,
                  PATH=f"{ENV / 'bin'}:{os.environ['PATH']}",
                  OMP_NUM_THREADS='2', PYTHONUNBUFFERED='1', TF_CPP_MIN_LOG_LEVEL='2',
                  MALLOC_ARENA_MAX='2', MALLOC_TRIM_THRESHOLD_='0')
os.chdir(REPO)
image = json.loads((ENV / '.image').read_text())
print('Image:', json.dumps(image, indent=1))
print('Model:', model_path or 'gs://gemma-data/checkpoints/gemma4-e2b-it (default)')

## 3. Verify the TPU and launch agy

The image's jax must see the TPU, and the agent must create and execute a Python file before the benchmark starts. Its transcript stays in this notebook.

In [ ]:
def run_visible(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)

run_visible([str(ENV / 'bin/python'), '-c',
             "import jax; assert jax.default_backend() == 'tpu', jax.default_backend(); "
             "print([(str(d), d.device_kind) for d in jax.devices()]); print('jax', jax.__version__)"])
print('Source revision:', image['commit'])
print('Runtime:', os.environ.get('COLAB_RELEASE_TAG', 'non-Colab environment'))
if not os.environ.get('GEMINI_API_KEY'):
    try:
        from google.colab import userdata
        os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    except Exception:
        import getpass
        os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')
assert os.environ['GEMINI_API_KEY']
agent_dir = DEMO_ROOT / 'agent_check'
run_visible([str(ENV / 'bin/python'), '-c', f"""
from pathlib import Path
from evalrx.agent_runtime.providers.antigravity import AntigravityAgent
agent_dir = Path({str(agent_dir)!r})
agent = AntigravityAgent(binary_path={str(ENV / 'bin/agy')!r}, model='gemini-3.8-flash-low',
                         timeout_sec=180, extra_args=['--output-format', 'stream-json'])
result = agent.run('Create agent_check.py that prints AGENT_READY. Run it with Python and report its output. Work only in this directory.', agent_dir)
print('Agent:', result.provider_name)
print(result.raw_output)
assert not result.error, result.error
assert (agent_dir / 'agent_check.py').exists()
print((agent_dir / 'agent_check.py').read_text())
"""])
run_visible([str(ENV / 'bin/python'), str(agent_dir / 'agent_check.py')])

## 4. Run the complete EvalRX workflow

The command runs baseline evaluation, M1 probes, agent-written exploratory analysis, M2 statistics, M3 hypotheses, M4 tests and M5 repair search. `--skip-surgery` omits the separate surgery experiments; M5 repair still runs. A run takes about an hour on a TPU v5e.

In [ ]:
%%bash
set -euo pipefail
"$EVALRX_ENV/bin/evalrx" run \
  --modality llm --model gemma-4-e2b --dataset "$EVALRX_DATASET" \
  --backend jax_local --device tpu --dtype bfloat16 \
  --limit 128 --download-limit 128 --seed 0 \
  --max-new-tokens 512 --temperature 0 --analyzer-max-cases 8 \
  --judge-provider agy --judge-model gemini-3.8-flash-low \
  --fix-tier L2 --fix-repair-rounds 2 --fix-validation-cases 64 \
  --allow-codegen --explore --m2-codegen --skip-surgery \
  --data-dir "$EVALRX_DEMO_ROOT/data" --run-dir "$EVALRX_DEMO_ROOT/runs" \
  --run-tag "$EVALRX_RUN_TAG" \
  ${EVALRX_MODEL_PATH:+--model-path "$EVALRX_MODEL_PATH"}

## 5. Read the measured result

The benchmark uses its registered exact-answer grader. EXPLORE chooses the candidate; only an independent CONFIRM result can establish a successful repair.

In [ ]:
import html
from IPython.display import display, HTML, Markdown, Image, JSON

def table(rows, columns=None):
    """An HTML table from a list of dicts (no pandas: the kernel's packages are not used)."""
    columns = columns or list(dict.fromkeys(k for row in rows for k in row))
    head = ''.join(f'<th>{html.escape(str(c))}</th>' for c in columns)
    body = ''.join('<tr>' + ''.join(f'<td style="text-align:left;white-space:pre-wrap">{html.escape(str(row.get(c, "")))}</td>'
                                    for c in columns) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))

run_dir = DEMO_ROOT / f'runs/gemma-4-e2b/{DATASET}.{RUN_TAG}'
read_json = lambda p: json.loads(p.read_text())
baseline = read_json(run_dir / 'baseline.json')
summary = read_json(run_dir / 'summary.json')
stages = {stage: read_json(run_dir / 'logs' / stage / 'log.json')
          for stage in ['M1', 'M2', 'M3', 'M4', 'M5']
          if (run_dir / 'logs' / stage / 'log.json').exists()}
fix = (stages.get('M5', {}).get('fix') or [{}])[-1]
attempts = fix.get('attempted', [])
print(f"Baseline: {baseline['n_pass']}/{baseline['n']} ({baseline['accuracy']:.1%})")
print(f"Verified hypotheses: {summary['n_verified']}")
print('Frozen candidate:', fix.get('selected_on_explore') or 'none')
if attempts:
    final = attempts[0]
    table([{
        'split': 'CONFIRM', 'n': final['n_pairs'],
        'baseline_correct': final['n_baseline_correct'],
        'candidate_correct': final['n_candidate_correct'],
        'repaired_cases': final['n_fixed'], 'regressed_cases': final['n_broken'],
        'e_value': final['e_value'], 'required_e_value': final['e_threshold'],
        'verdict': final['verdict'], 'validated_fix': final['fixed'],
    }])
    display(JSON(final['payload']))
    print('Accuracy repair validated.' if final['fixed'] else 'No statistically confirmed accuracy repair.')
else:
    print('No candidate selected for CONFIRM; no validated accuracy repair.')
if fix.get('selection_attempted'):
    display(Markdown('### EXPLORE candidates (selection only)'))
    table(fix['selection_attempted'])
for image_file in sorted((run_dir / 'logs/M2/artifacts').glob('*.png')):
    display(Image(filename=str(image_file)))

## 6. Inspect every answer

In [ ]:
manifest = read_json(DEMO_ROOT / f'data/{DATASET}/manifest.json')
questions = {row['id']: row['prompt'] for row in manifest}
table([{**row, 'question': questions[row['id']]} for row in baseline['cases']],
      ['id', 'question', 'expected', 'observed', 'label'])
if attempts:
    before = {row['id']: row for row in baseline['cases']}
    table([{
        'id': cid, 'expected': before[cid]['expected'],
        'baseline': before[cid]['observed'], 'candidate': answer,
        'repaired': cid in final['fixed_cases'], 'regressed': cid in final['broken_cases'],
    } for cid, answer in final['outputs'].items()])

## 7. Full agent and stage records

The saved records include generated code, analysis, hypotheses, candidate selection and confirmation. Save the executed notebook to keep these results together.

In [ ]:
display(JSON({'baseline': baseline, 'summary': summary, 'stages': stages}))
for source in sorted(run_dir.rglob('*.py')):
    print('\n---', source.relative_to(run_dir), '---')
    print(source.read_text())